# CYTools-agent

An agent that drives CYTools (fetch polytopes, triangulate, build CYs) with a local Ollama model.

Run with the **Python (cytools-agent)** kernel. Run the **Setup** cells once, then **chat** -- add `agent.chat(...)` cells freely; the agent remembers earlier turns. Re-run the *Start a session* cell to reset.

## Setup (run once)

In [ ]:
from openai import OpenAI
import os

base = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
client = OpenAI(base_url=base + "/v1", api_key="ollama")
MODEL = "qwen3:8b"
assert MODEL in [m.id for m in client.models.list().data], f"{MODEL} not pulled"
print("OK:", MODEL)

In [2]:
from cytools_agent.tools import (polytope, triangulation, cy, code)
from cytools_agent.schema import function_to_schema

TOOL_FNS = [polytope.fetch_polytopes, polytope.get_polytope_info,
            polytope.ks_stats,
            triangulation.get_heights,
            triangulation.get_triangulation_info,
            cy.get_cy_info, cy.get_cy_cones,
            code.run_python, code.cytools_help]
tools = [function_to_schema(fn) for fn in TOOL_FNS]
tool_impls = {fn.__name__: fn for fn in TOOL_FNS}

In [3]:
from cytools_agent.agent import Agent
from cytools_agent.prompt import DEFAULT_SYSTEM_PROMPT as system_prompt
# (edit system_prompt here to customize)

## Start a session

Run this to begin -- or re-run it to reset the conversation.

In [4]:
agent = Agent(client, MODEL, system_prompt, tools, tool_impls,
              max_steps=20, verbosity=2)

## Chat

Add more `agent.chat(...)` cells below; the agent remembers earlier turns.

In [5]:
print(agent.chat("Fetch 3 polytopes at h11=5"))

Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_pgxyfp2q', function=Function(arguments='{"h11":5}', name='ks_stats'), type='function', index=0)]`
Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_p7dlctg6', function=Function(arguments='{"h11":5,"limit":3}', name='fetch_polytopes'), type='function', index=0)]`
Agent: Text message `ChatCompletionMessage(content="I fetched 3 polytopes at h11=5 with the following ids: `['h11-5_h21-20_ind-0', 'h11-5_h21-29_ind-0', 'h11-5_h21-29_ind-1']`.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)`
I fetched 3 polytopes at h11=5 with the following ids: `['h11-5_h21-20_ind-0', 'h11-5_h21-29_ind-0', 'h11-5_h21-29_ind-1']`.


In [7]:
print(agent.chat("How many NTFEs do each of them have?"))

Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_np51y49u', function=Function(arguments='{"ks_ind":"h11-5_h21-20_ind-0","kind":"NTFE"}', name='get_heights'), type='function', index=0)]`
Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_4peqy1ci', function=Function(arguments='{"kind":"NTFE","ks_ind":"h11-5_h21-29_ind-0"}', name='get_heights'), type='function', index=0)]`
Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_zafte5o9', function=Function(arguments='{"kind":"NTFE","ks_ind":"h11-5_h21-29_ind-1"}', name='get_heights'), type='function', index=0)]`
Agent: Text message `ChatCompletionMessage(content='The three polytopes have the following NTFE counts (number of inequivalent triangulations):\n\n1. `h11-5_h21-20_ind-0`: **2** NTFEs  \n2. `h11-5_h21-29_ind-0`: **1** NTFE  \n3. `h11-5_h21-29_ind-1`: **1** NTFE  \n\nThese counts are derived from the `shape[0]` value of their `get_heights` results.', refusal=None, role='assistant', annotat

In [8]:
print(agent.chat("What are the CY volumes at the tip of the stretched kahler cone for each of their associated CYs?"))

Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_qvrcga8w', function=Function(arguments='{"cone":"Kcup","heights":[0,0,0,0,0,1,0,0,0,0],"ks_ind":"h11-5_h21-20_ind-0","t":"tip"}', name='get_cy_info'), type='function', index=0)]`
Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_k3akan9s', function=Function(arguments='{"cone":"Kcup","heights":[0,0,0,0,0,-1,0,0,0,0],"ks_ind":"h11-5_h21-20_ind-0","t":"tip"}', name='get_cy_info'), type='function', index=0)]`
Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_pwaw7gr6', function=Function(arguments='{"ks_ind":"h11-5_h21-29_ind-0","t":"tip","cone":"Kcup","heights":[0,1,1,0,1,0,0,0,0]}', name='get_cy_info'), type='function', index=0)]`
Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_0l271s9s', function=Function(arguments='{"h11":5,"favorable":true}', name='ks_stats'), type='function', index=0)]`
Agent: Tool call `[ChatCompletionMessageFunctionToolCall(id='call_7q6c060x', function

In [9]:
# Save the session as a standalone Python script:
# agent.save_history("session.py")

## Orchestrator (PM + engineer)

For bigger multi-step questions (loop over many polytopes, compute, plot), use the **orchestrator** instead of `agent.chat`: a project-manager model plans the work and a separate engineer executes it step by step, with every result captured in an evidence log.

One call does everything -- no setup beyond the imports:

```python
from cytools_agent.orchestrator import run_session
print(run_session("your question", model=MODEL))
```

**Watch it live:** run `python -m cytools_agent.viewer` in a terminal and open http://127.0.0.1:8765 -- the plan, each engineer step (code + real output), and figures render as the session runs. Finished sessions are archived in `scratch/logs/` and browsable in the same viewer; `python -m cytools_agent.viewer export` bakes one into a shareable standalone HTML.

In [10]:
from cytools_agent.orchestrator import run_session

print(run_session(
    "For the first 10 polytopes at each h11 from 2 through 10, make a scatter of "
    "automorphism group order vs lattice point count colored by h11 and a "
    "histogram of NTFE triangulation counts; report the mean NTFE count and the "
    "id of the most symmetric polytope.",
    model=MODEL))


[PM direct speech]
For each h11 from 2 to 10, take the first 10 polytopes, and make a scatter plot showing the size of their automorphism groups against the number of lattice points, using different colors for each h11. Also, create a histogram showing how many NTFE triangulations each polytope has. Finally, report the average number of NTFE triangulations and the ID of the polytope with the largest automorphism group.

[pipeline spec]
{'fits': True, 'fetch': {'h11': [2, 3, 4, 5, 6, 7, 8, 9, 10], 'limit': 10}, 'map': {'automorphism_order': "get_polytope_info(ks_ind)['automorphism_order']", 'n_points': "get_polytope_info(ks_ind)['n_points']", 'ntfe_count': "get_heights(ks_ind)['shape'][0]"}, 'reduce': [{'name': 'ntfe_avg', 'op': 'mean', 'of': 'ntfe_count'}, {'name': 'max_automorphism_id', 'op': 'argmax', 'of': 'automorphism_order'}], 'search': None, 'plot': [{'kind': 'scatter', 'x': 'n_points', 'y': 'automorphism_order', 'color': None, 'logx': True, 'logy': True}, {'kind': 'histogram',

In [ ]:
print(run_session(
    "Fetch the first 10x polytopes at each h11 in [2,10] and "
    "plot the size of their automorphism group vs h11",
    model=MODEL))

In [ ]:
# For research questions where you want extra confidence: run several
# independent sessions and accept the answer only when the final numbers
# agree (~votes x the wall clock; disagreement is flagged LOW CONFIDENCE).
# from cytools_agent.orchestrator import run_session_voted
# print(run_session_voted("your question", votes=3, model=MODEL))

### Conversational orchestrator

`OrchestratorChat` keeps state between questions: the polytopes and columns one turn computes stay available, so follow-ups like *"those same polytopes"* or *"now plot that vs h21"* just work. Plots can sweep h11 ranges, color points by a third quantity, overlay histograms by category, and use log axes -- and you can ask for several figures in one question.

In [ ]:
from cytools_agent.orchestrator import OrchestratorChat

ochat = OrchestratorChat(model=MODEL, verbose=False)
print(ochat.chat("Fetch the first 25 polytopes at h11=3 and report how many "
                 "NTFE triangulations each has."))

In [ ]:
print(ochat.chat("Now scatter those NTFE counts against the polytopes' h21 "
                 "values."))